In [2]:
## Initialzing and loading required libraries and subfunctions
import numpy as np
import matplotlib.pyplot as plt
import copy
import yasa
from mne.filter import resample
import pynapple as nap
import seaborn as sns
import pandas as pd
from sklearn.preprocessing import normalize
import requests
from io import BytesIO
import sails
import re
from scipy.stats import entropy
import os, sys

import scipy
from scipy import signal
from scipy.interpolate import griddata
from scipy.signal import correlate
from scipy.stats import pearsonr
from scipy.fft import fft
from scipy.spatial.distance import euclidean
from scipy.signal import spectrogram
from scipy.io import loadmat
import scipy.fft
import scipy.stats
import scipy.io as sio
from scipy.signal import hilbert

import emd as emd
import emd.sift as sift
import emd.spectra as spectra

from neurodsp.sim import sim_combined
from neurodsp.plts import plot_time_series, plot_timefrequency
from neurodsp.utils import create_times
from neurodsp.timefrequency.wavelets import compute_wavelet_transform
from neurodsp.filt import filter_signal

# Load required libraries
import numpy as np
from scipy.io import loadmat
from scipy.signal import hilbert
from scipy.interpolate import interp1d
import matplotlib.pyplot as plt
import seaborn as sns
from neurodsp.filt import filter_signal, filter_signal_fir, design_fir_filter
import emd
import pandas as pd
from sklearn.preprocessing import Normalizer
from tqdm import tqdm
import plotly.express as px
import copy
import umap.umap_ as umap
import skdim
from scipy.spatial import cKDTree
import pickle
from mpl_toolkits.axes_grid1 import make_axes_locatable
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
from matplotlib.cm import ScalarMappable

## UTILS

for rel in ('src', '../src'):
    p = os.path.abspath(os.path.join(os.getcwd(), rel))
    if os.path.isdir(p) and p not in sys.path:
        sys.path.insert(0, p)

from utils import *
from detect_pt import *

from scipy.io import loadmat
import numpy as np
from neurodsp.filt import filter_signal
import copy
import emd
from scipy.spatial import cKDTree
from tqdm import tqdm

sns.set(style='white', context='notebook')

In [3]:
path_to_config = '/Users/amir/Desktop/for Abdel/thetaGamma_dev/src/emd_masksift_CA1_config_2500.yml'
config = emd.sift.SiftConfig.from_yaml_file(path_to_config)

In [4]:
def extract_cycle_info(imfs, imf_frequencies):

  waveforms = pd.DataFrame()
  all_trials = pd.DataFrame()
  raw_wavelets = []
  all_FPPs = []

  theta_range = [5, 12]
  frequencies = np.arange(15, 141, 1)
  angles=np.linspace(-180,180,19)
  fs = 2500

  for idx, imf in enumerate(imfs):
    cycle_data = get_cycle_data(imf[:, 5], fs=2500)

    amp_thresh = np.percentile(cycle_data['IA'], 25) # higher than 25th percentile of the data
    lo_freq_duration = fs/5  # restrict the analysis to 5-12 Hz
    hi_freq_duration = fs/12

    conditions = ['is_good==1',
                        f'duration_samples<{lo_freq_duration}',
                        f'duration_samples>{hi_freq_duration}',
                        f'max_amp>{amp_thresh}']
    print(len(cycle_data['theta_imf']))
    all_cycles = get_cycles_with_conditions(cycle_data['cycles'], conditions)
    
    # Check if any cycles satisfy the conditions
    if all_cycles is None or all_cycles.chain_vect.size == 0:
        print("No cycles satisfy the conditions.")
        return pd.DataFrame(), pd.DataFrame(), []
    
    subset_cycles_df = all_cycles.get_metric_dataframe(subset=True)
    subset_indices = subset_cycles_df['index'].values

    all_cycles_inds = get_cycle_inds(all_cycles, subset_indices)
    cycles_inds = arrange_cycle_inds(all_cycles_inds)

    freqs = imf_frequencies[idx]
    sub_theta, theta, supra_theta = tg_split(freqs, theta_range)
    supra_theta_sig = np.sum(imf.T[supra_theta], axis=0)

    # # Corrected Wavelet Transform Computation
    raw_data=sails.wavelet.morlet(supra_theta_sig, freqs=frequencies, sample_rate=fs, ncycles=5,ret_mode='power', normalise=None)
    raw_wavelets.append(raw_data)
    supraPlot = scipy.stats.zscore(raw_data, axis=1)
    FPP = bin_tf_to_fpp(cycles_inds, supraPlot, bin_count=19)
    all_FPPs.append(FPP)

    # Compute mode frequency for each cycle
    mode_freqs, entropies = compute_mode_frequency_and_entropy(FPP, frequencies, angles)

    all_waveforms, _ = emd.cycles.phase_align(cycle_data['IP'], cycle_data['theta_imf'],
                                                            cycles=all_cycles.iterate(through='subset'), npoints=100)
    all_waveforms = pd.DataFrame(all_waveforms.T)

    waveforms = pd.concat([waveforms, all_waveforms])

    trial = all_cycles.get_metric_dataframe(subset=True)
    trial['mode_freqs'] = mode_freqs
    trial['entropy'] = entropies
    all_trials = pd.concat([all_trials, trial])

  return waveforms, all_trials, all_FPPs

# pre-processing

In [5]:
# =============================================================================
# ========== FPP-derived spectral signatures over the whole dataset ===========
# =============================================================================
import numpy as np
import pandas as pd
import os, re, warnings
import scipy.signal as spsig
from scipy.stats import zscore
import emd

# ------------- Morlet amplitude CWT (same as preprocess v2 path) -------------
def _morlet_amp_spectrogram(x, fs, freqs_hz, w=6.0):
    """
    Morlet CWT amplitude (|complex CWT|), using scipy.signal.cwt + morlet2.
    Returns array [n_freq, n_time].
    """
    freqs_hz = np.asarray(freqs_hz, float)
    scales = (w * fs) / (2.0 * np.pi * freqs_hz)  # s = w*fs/(2π f)
    cwt_mat = spsig.cwt(x, spsig.morlet2, scales, w=w)
    return np.abs(cwt_mat)

# -------------------------- Theta IMF picker (v2-ish) -------------------------
def _choose_theta_imf(imf, fs, imf_centers_hz, theta_band=(5.0, 12.0), prefer_idx=5):
    """
    Prefer a fixed theta IMF index if available; otherwise pick IMF with center
    frequency closest to the theta band center.
    """
    if imf.shape[1] > prefer_idx:
        return prefer_idx
    lo, hi = theta_band
    centers = np.asarray(imf_centers_hz).astype(float)
    if centers.ndim == 1 and centers.size == imf.shape[1]:
        band_center = 0.5*(lo + hi)
        return int(np.argmin(np.abs(centers - band_center)))
    warnings.warn("Unexpected imf_centers_hz shape; defaulting to last IMF.")
    return imf.shape[1] - 1

# --------------------------- Cycle bounds utility -----------------------------
def _cycle_bounds_from_inds(all_cycles_inds):
    """
    Convert per-cycle index arrays -> [start, end] bounds for each cycle.
    """
    bounds = []
    for cyc in all_cycles_inds:
        cyc = np.asarray(cyc)
        cyc = cyc[np.isfinite(cyc)]
        if cyc.size > 1:
            s, e = int(cyc[0]), int(cyc[-1])
            if e > s:
                bounds.append([s, e])
    return np.array(bounds, dtype=int)

# ---------------- FPP-derived spectral signatures for ONE segment -------------
def spectral_signatures_from_fpp_for_segment(
    imf, imf_centers_hz, fs,
    theta_band=(5,12),
    freq_vec=np.arange(15,141,1),
    morlet_w=6.0,
    n_bins=19,
    normalize='zscore',   # 'zscore' or 'none'
):
    """
    Returns:
      sigs_per_cycle : (n_cycles, n_freq) spectral signatures (FPP-based)
      n_cycles       : int, number of valid cycles used
    """
    # 1) theta IMF & cycle selection (same filters you use)
    theta_idx = _choose_theta_imf(imf, fs, imf_centers_hz, theta_band=theta_band, prefer_idx=5)
    cycle_data = get_cycle_data(imf[:, theta_idx], fs=fs)
    if cycle_data is None or 'IA' not in cycle_data:
        return np.zeros((0, len(freq_vec))), 0

    amp_thresh = np.percentile(cycle_data['IA'], 25)
    lo_dur = fs / float(theta_band[0])  # 5 Hz -> shortest period
    hi_dur = fs / float(theta_band[1])  # 12 Hz -> longest period
    conditions = [
        'is_good==1',
        f'duration_samples<{lo_dur}',
        f'duration_samples>{hi_dur}',
        f'max_amp>{amp_thresh}',
    ]
    all_cycles = get_cycles_with_conditions(cycle_data['cycles'], conditions)
    if all_cycles is None or getattr(all_cycles, 'chain_vect', np.array([])).size == 0:
        return np.zeros((0, len(freq_vec))), 0

    subset_df = all_cycles.get_metric_dataframe(subset=True)
    subset_indices = subset_df['index'].values
    all_cycles_inds = get_cycle_inds(all_cycles, subset_indices)
    cycle_bounds = _cycle_bounds_from_inds(all_cycles_inds)
    if cycle_bounds.size == 0:
        return np.zeros((0, len(freq_vec))), 0

    # 2) supra-theta signal (sum of IMFs > 12 Hz)
    sub_mask, theta_mask, supra_mask = tg_split(imf_centers_hz, theta_band)
    supra_theta_sig = imf[:, supra_mask].sum(axis=1) if np.any(supra_mask) else np.zeros(imf.shape[0])

    # 3) Morlet amplitude TFR (15–140 Hz) and optional normalization
    tf_amp = _morlet_amp_spectrogram(supra_theta_sig, fs, freq_vec, w=morlet_w)  # [n_freq, n_time]
    if normalize == 'zscore':
        tf_use = zscore(tf_amp, axis=1)  # per-freq z across time
    elif normalize == 'none':
        tf_use = tf_amp
    else:
        raise ValueError("normalize must be 'zscore' or 'none'.")

    # 4) Per-cycle FPP -> per-cycle spectral signature (mean across phase bins)
    sigs = []
    for b in cycle_bounds:
        # fpp_single: [1, n_freq, n_bins]  -> squeeze -> [n_freq, n_bins]
        fpp_single = bin_tf_to_fpp(b, tf_use, bin_count=n_bins)
        fpp_single = np.squeeze(fpp_single, axis=0)
        sig_single = np.nanmean(fpp_single, axis=1)  # collapse phase bins
        sigs.append(sig_single)

    if len(sigs) == 0:
        return np.zeros((0, len(freq_vec))), 0
    return np.vstack(sigs), len(sigs)

# ---------------------- Dataset-level extractor (per rat) ----------------------
def extract_data_for_rat_fppsig(rat_id):
    """
    Mirrors your extract_data_for_rat(), but collects FPP-derived spectral signatures.

    Returns:
      all_combined_waveforms : pd.DataFrame
      all_combined_trials    : pd.DataFrame
      all_phasic_fpps        : list (kept from your original, if you still use it upstream)
      all_tonic_fpps         : list
      all_phasic_time_sigs   : list of arrays [(n_cycles_i, n_freq), ...]  <-- FPP-derived
      all_tonic_time_sigs    : list of arrays [(n_cycles_i, n_freq), ...]  <-- FPP-derived
    """
    base_path = '/Users/amir/Desktop/for Abdel/OS Basic'
    fs = 2500

    all_combined_waveforms = pd.DataFrame()
    all_combined_trials    = pd.DataFrame()
    all_phasic_fpps, all_tonic_fpps = [], []  # unchanged placeholders (if you still need them)

    all_phasic_time_sigs, all_tonic_time_sigs = [], []

    rat_path = os.path.join(base_path, str(rat_id))
    if not os.path.isdir(rat_path):
        print(f"Rat folder {rat_id} does not exist.")
        return None, None

    recording_folders = [f for f in os.listdir(rat_path) if os.path.isdir(os.path.join(rat_path, f))]
    if not recording_folders:
        print(f"No recording folders found for Rat {rat_id}.")
        return None, None

    for recording_folder in recording_folders:
        print(f"Processing recording folder: {recording_folder}")
        recording_path = os.path.join(rat_path, recording_folder)

        match = re.match(r'^Rat-OS-Ephys_(Rat\d+)_SD(\d+)_([\w-]+)_([\d-]+)$', recording_folder)
        if not match:
            print(f"Unexpected folder name format: {recording_folder}. Skipping...")
            continue

        rat_id_part = match.group(1)
        sd_number   = match.group(2)
        condition   = match.group(3)
        date_part   = match.group(4)
        rat_id_from_folder = ''.join(filter(str.isdigit, rat_id_part))
        if rat_id_from_folder != str(rat_id):
            print(f"Rat ID mismatch in folder {recording_folder}. Expected Rat{rat_id}, found Rat{rat_id_from_folder}. Skipping...")
            continue

        trial_folders = [
            f for f in os.listdir(recording_path)
            if os.path.isdir(os.path.join(recording_path, f)) and
               re.search(r'(?i)post[\-_]?trial[\-_]?([2-5])', f)
        ]
        if not trial_folders:
            print(f"No trial folders found in {recording_folder}.")
            continue

        for trial_folder in trial_folders:
            print(f"  Processing trial folder: {trial_folder}")
            trial_path = os.path.join(recording_path, trial_folder)

            lfp_file, state_file = None, None
            for file_name in os.listdir(trial_path):
                if 'HPC' in file_name and file_name.endswith('.mat'):
                    lfp_file = os.path.join(trial_path, file_name)
                elif 'states' in file_name and file_name.endswith('.mat'):
                    state_file = os.path.join(trial_path, file_name)
                elif 'States' in file_name and file_name.endswith('.mat'):
                    state_file = os.path.join(trial_path, file_name)
            if not lfp_file or not state_file:
                print(f"    Missing LFP or state file in {trial_path}. Skipping...")
                continue

            trial_number_match = re.search(r'(?i)post[\-_]?trial[\-_]?([2-5])', trial_folder)
            if not trial_number_match:
                print(f"    Unable to extract trial number from folder name: {trial_folder}. Skipping...")
                continue
            trial_number = int(trial_number_match.group(1))

            # -------- Load data & get phasic/tonic intervals --------
            try:
                lfpHPC, hypno, _ = get_data(lfp_file, state_file)
                try:
                    phasic_interval, tonic_interval, lfp_raw = extract_pt_intervals(lfpHPC, hypno)
                except ValueError:
                    print(f"    No REM sleep found in {trial_folder} (Rat {rat_id}, Condition {condition}). Filling with empty intervals.")
                    phasic_interval, tonic_interval, lfp_raw = [[], [], []]

                if not (phasic_interval and tonic_interval):
                    continue

                # -------- Extract IMFs + frequencies for both REM types --------
                tonic_imfs,  tonic_freqs,  tonic_lpf  = extract_imfs_by_pt_intervals(
                    lfp_raw, fs, tonic_interval,  config, return_imfs_freqs=True)
                phasic_imfs, phasic_freqs, phasic_lpf = extract_imfs_by_pt_intervals(
                    lfp_raw, fs, phasic_interval, config, return_imfs_freqs=True)

                # ====== Your existing FPP path (kept if you still need upstream artifacts) ======
                phasic_waveforms, phasic_trials, phasic_fpps = extract_cycle_info(phasic_imfs, phasic_freqs)
                tonic_waveforms,  tonic_trials,  tonic_fpps = extract_cycle_info(tonic_imfs,  tonic_freqs)
                all_phasic_fpps.extend(phasic_fpps)
                all_tonic_fpps.extend(tonic_fpps)

                # ====== NEW: FPP-derived spectral signatures (per-cycle) for BOTH REM types ======
                # Each call returns a list of arrays: one (n_cycles x n_freq) per REM segment
                # We collect them flat like in your v2 helper
                for seg_imf, seg_freqs in zip(phasic_imfs, phasic_freqs):
                    sigs, ncyc = spectral_signatures_from_fpp_for_segment(
                        seg_imf, seg_freqs, fs,
                        theta_band=(5,12),
                        freq_vec=np.arange(15,141,1),
                        morlet_w=6.0,
                        n_bins=19,
                        normalize='zscore'  # or 'none' if you prefer raw amplitude
                    )
                    all_phasic_time_sigs.append(sigs)

                for seg_imf, seg_freqs in zip(tonic_imfs, tonic_freqs):
                    sigs, ncyc = spectral_signatures_from_fpp_for_segment(
                        seg_imf, seg_freqs, fs,
                        theta_band=(5,12),
                        freq_vec=np.arange(15,141,1),
                        morlet_w=6.0,
                        n_bins=19,
                        normalize='zscore'  # or 'none'
                    )
                    all_tonic_time_sigs.append(sigs)

                # -------- Add metadata columns to your dataframes --------
                for df in [phasic_waveforms, phasic_trials]:
                    df['rat_id']    = rat_id
                    df['condition'] = condition
                    df['trial']     = trial_number
                    df['cycle_type']= 'phasic'
                    df['SD']        = sd_number
                    df['date']      = date_part
                for df in [tonic_waveforms, tonic_trials]:
                    df['rat_id']    = rat_id
                    df['condition'] = condition
                    df['trial']     = trial_number
                    df['cycle_type']= 'tonic'
                    df['SD']        = sd_number
                    df['date']      = date_part

                all_combined_waveforms = pd.concat(
                    [all_combined_waveforms, phasic_waveforms, tonic_waveforms], ignore_index=True)
                all_combined_trials = pd.concat(
                    [all_combined_trials, phasic_trials, tonic_trials], ignore_index=True)

            except FileNotFoundError:
                print(f"    Data not found in {trial_path}. Skipping...")

    if all_combined_waveforms.empty:
        print(f"No data extracted for Rat {rat_id}.")
        return None, None

    return (all_combined_waveforms, all_combined_trials,
            all_phasic_fpps, all_tonic_fpps,
            all_phasic_time_sigs, all_tonic_time_sigs)

## single rat preprocess

In [6]:
rat_id = '1'
waveforms_df, trials_df, all_phasic_FPPs, all_tonic_FPPs, phasic_time_signatures, tonic_time_signatures= extract_data_for_rat_fppsig(rat_id)

print(f"FPPs — phasic intervals: {len(all_phasic_FPPs)}, tonic intervals: {len(all_tonic_FPPs)}")
print(f"Time-mean signatures — phasic intervals: {len(phasic_time_signatures)}, tonic intervals: {len(tonic_time_signatures)}")

Processing recording folder: Rat-OS-Ephys_Rat1_SD1_OD_21-09-2017
  Processing trial folder: post_trial3_2017-09-21_12-48-10
There was 0 in the dataset
Number of detected Tonic intrevals:18
Number of detected Tonic intrevals after threshold:18
 Checking Cycles inputs - trimming singleton from input 'IP'
6635
 Checking phase_align inputs - trimming singleton from input 'ip'
 Checking Cycles inputs - trimming singleton from input 'IP'
4360
 Checking phase_align inputs - trimming singleton from input 'ip'
 Checking Cycles inputs - trimming singleton from input 'IP'
3735
 Checking phase_align inputs - trimming singleton from input 'ip'
 Checking Cycles inputs - trimming singleton from input 'IP'
3839
 Checking phase_align inputs - trimming singleton from input 'ip'
 Checking Cycles inputs - trimming singleton from input 'IP'
4275
 Checking phase_align inputs - trimming singleton from input 'ip'
 Checking Cycles inputs - trimming singleton from input 'IP'
7665
 Checking phase_align inputs - 

## all rats preprocess

In [3]:
import pickle
import pandas as pd
from pathlib import Path

# ===== CONFIG =====
rat_ids  = [1, 3, 4, 6, 9, 11, 13]
out_root = Path("./fppsig_v3_outputs")
out_root.mkdir(parents=True, exist_ok=True)

# toggle between saving separately or as one pickle
save_all_together = False   # ← True = all_objects.pkl, False = waveforms_df + trials_df + lists.pkl

def _save_df(df: pd.DataFrame, path_no_ext: Path) -> str:
    """Save DataFrame as Parquet if possible; else CSV. Return saved path."""
    try:
        p = path_no_ext.with_suffix(".parquet")
        df.to_parquet(str(p), index=False)
        return str(p)
    except Exception:
        p = path_no_ext.with_suffix(".csv")
        df.to_csv(str(p), index=False)
        return str(p)

manifest_rows = []

for rid in rat_ids:
    print(f"\n=== Rat {rid} ===")
    res = extract_data_for_rat_fppsig(str(rid))
    if not isinstance(res, tuple) or len(res) != 6:
        print(f"  Skipping Rat {rid}: unexpected return value.")
        continue

    (waveforms_df,
     trials_df,
     all_phasic_FPPs,
     all_tonic_FPPs,
     phasic_time_signatures,
     tonic_time_signatures) = res

    if waveforms_df is None or trials_df is None:
        print(f"  Skipping Rat {rid}: no data.")
        continue

    rat_dir = out_root / f"Rat{rid}"
    rat_dir.mkdir(parents=True, exist_ok=True)

    if save_all_together:
        # ----- one combined file -----
        all_path = rat_dir / "all_objects.pkl"
        with open(all_path, "wb") as f:
            pickle.dump(
                dict(
                    waveforms_df=waveforms_df,
                    trials_df=trials_df,
                    all_phasic_FPPs=all_phasic_FPPs,
                    all_tonic_FPPs=all_tonic_FPPs,
                    phasic_time_signatures=phasic_time_signatures,
                    tonic_time_signatures=tonic_time_signatures
                ),
                f,
                protocol=pickle.HIGHEST_PROTOCOL
            )
        wf_path = tr_path = lists_path = None
        print(f"  Saved everything → {all_path}")

    else:
        # ----- separate files -----
        wf_path = _save_df(waveforms_df, rat_dir / "waveforms_df")
        tr_path = _save_df(trials_df,   rat_dir / "trials_df")

        lists_path = rat_dir / "lists.pkl"
        with open(lists_path, "wb") as f:
            pickle.dump(
                dict(
                    all_phasic_FPPs=all_phasic_FPPs,
                    all_tonic_FPPs=all_tonic_FPPs,
                    phasic_time_signatures=phasic_time_signatures,
                    tonic_time_signatures=tonic_time_signatures
                ),
                f,
                protocol=pickle.HIGHEST_PROTOCOL
            )
        print(f"  Saved separate files → {rat_dir}")

    # --- manifest info ---
    n_phasic_segments = len(phasic_time_signatures)
    n_tonic_segments  = len(tonic_time_signatures)
    n_phasic_cycles   = int(sum((arr.shape[0] for arr in phasic_time_signatures if hasattr(arr, "shape")), 0))
    n_tonic_cycles    = int(sum((arr.shape[0] for arr in tonic_time_signatures  if hasattr(arr, "shape")), 0))

    manifest_rows.append({
        "rat_id": rid,
        "n_phasic_segments": n_phasic_segments,
        "n_tonic_segments": n_tonic_segments,
        "n_phasic_cycles_total": n_phasic_cycles,
        "n_tonic_cycles_total": n_tonic_cycles,
        "waveforms_df_path": wf_path,
        "trials_df_path": tr_path,
        "lists_pickle_path": str(lists_path) if not save_all_together else None,
        "all_objects_pickle_path": str(all_path) if save_all_together else None
    })

# save manifest
manifest_df = pd.DataFrame(manifest_rows).sort_values("rat_id").reset_index(drop=True)
manifest_path = out_root / "manifest.csv"
manifest_df.to_csv(manifest_path, index=False)
print(f"\nManifest saved to: {manifest_path}")
print(manifest_df)


=== Rat 1 ===


NameError: name 'extract_data_for_rat_fppsig' is not defined

## load all rats preproccessed data

In [5]:
import pickle
import pandas as pd
from pathlib import Path

def load_rat_data(rid, out_root=Path("./fppsig_v3_outputs")):
    """Load all data objects for a single rat."""
    rat_dir = out_root / f"Rat{rid}"

    # Option 1: combined pickle
    all_pkl = rat_dir / "all_objects.pkl"
    if all_pkl.exists():
        with open(all_pkl, "rb") as f:
            return pickle.load(f)

    # Option 2: separate files
    def _load_df(name):
        for ext in [".parquet", ".csv"]:
            p = rat_dir / f"{name}{ext}"
            if p.exists():
                return pd.read_parquet(p) if ext == ".parquet" else pd.read_csv(p)
        return None

    waveforms_df = _load_df("waveforms_df")
    trials_df = _load_df("trials_df")

    lists_pkl = rat_dir / "lists.pkl"
    if lists_pkl.exists():
        with open(lists_pkl, "rb") as f:
            lists = pickle.load(f)
    else:
        lists = {}

    return dict(waveforms_df=waveforms_df, trials_df=trials_df, **lists)


def load_all_rats(out_root=Path("./fppsig_v3_outputs")):
    """Load all rats listed in manifest.csv into a dict."""
    manifest_path = out_root / "manifest.csv"
    if not manifest_path.exists():
        raise FileNotFoundError(f"Manifest not found at {manifest_path}")

    manifest = pd.read_csv(manifest_path)
    rat_ids = manifest["rat_id"].tolist()

    all_data = {}
    for rid in rat_ids:
        print(f"Loading Rat {rid} ...", end=" ")
        try:
            all_data[rid] = load_rat_data(rid, out_root)
            print("✅ done")
        except Exception as e:
            print(f"⚠️ failed ({e})")

    return all_data

In [6]:
all_rats = load_all_rats()

Loading Rat 1 ... ✅ done
Loading Rat 3 ... ✅ done
Loading Rat 4 ... ✅ done
Loading Rat 6 ... ✅ done
Loading Rat 9 ... ✅ done
Loading Rat 11 ... ✅ done
Loading Rat 13 ... ✅ done


## load single rat preproccessed data

In [18]:
import pickle
import pandas as pd
from pathlib import Path

def load_one_rat(rid, out_root=Path("./fppsig_v3_outputs")):
    """
    Load all data objects for a single rat.
    Includes:
        - waveforms_df
        - trials_df
        - all_phasic_FPPs
        - all_tonic_FPPs
        - phasic_time_signatures
        - tonic_time_signatures

    Returns:
        all_data (dict)
        waveforms_df, trials_df, all_phasic_FPPs, all_tonic_FPPs,
        phasic_time_signatures, tonic_time_signatures
    """
    rat_dir = out_root / f"Rat{rid}"

    if not rat_dir.exists():
        raise FileNotFoundError(f"Directory not found for Rat {rid}: {rat_dir}")

    # Option 1: combined pickle
    all_pkl = rat_dir / "all_objects.pkl"
    if all_pkl.exists():
        with open(all_pkl, "rb") as f:
            print(f"Loading combined pickle for Rat {rid} ... ✅")
            all_data = pickle.load(f)
    else:
        # Option 2: separate files
        def _load_df(name):
            for ext in [".parquet", ".csv"]:
                p = rat_dir / f"{name}{ext}"
                if p.exists():
                    print(f"Loading {name}{ext} for Rat {rid} ... ✅")
                    return pd.read_parquet(p) if ext == ".parquet" else pd.read_csv(p)
            print(f"⚠️ {name} file not found for Rat {rid}")
            return None

        waveforms_df = _load_df("waveforms_df")
        trials_df = _load_df("trials_df")

        # Load any additional pickled lists or arrays
        lists_pkl = rat_dir / "lists.pkl"
        if lists_pkl.exists():
            with open(lists_pkl, "rb") as f:
                print(f"Loading lists.pkl for Rat {rid} ... ✅")
                lists = pickle.load(f)
        else:
            print(f"⚠️ lists.pkl not found for Rat {rid}")
            lists = {}

        # Combine all loaded data
        all_data = dict(
            waveforms_df=waveforms_df,
            trials_df=trials_df,
            **lists
        )

    print(f"✅ Finished loading Rat {rid}")

    # --- Separate specific objects for convenience ---
    waveforms_df = all_data.get("waveforms_df")
    trials_df = all_data.get("trials_df")
    all_phasic_FPPs = all_data.get("all_phasic_FPPs")
    all_tonic_FPPs = all_data.get("all_tonic_FPPs")
    phasic_time_signatures = all_data.get("phasic_time_signatures")
    tonic_time_signatures = all_data.get("tonic_time_signatures")

    # Return everything
    return (
        all_data,
        waveforms_df,
        trials_df,
        all_phasic_FPPs,
        all_tonic_FPPs,
        phasic_time_signatures,
        tonic_time_signatures,
    )

In [24]:
(   _,
    waveforms_df,
    trials_df,
    all_phasic_FPPs,
    all_tonic_FPPs,
    phasic_time_signatures,
    tonic_time_signatures,
) = load_one_rat("11")

Loading waveforms_df.parquet for Rat 11 ... ✅
Loading trials_df.parquet for Rat 11 ... ✅
Loading lists.pkl for Rat 11 ... ✅
✅ Finished loading Rat 11


# running tSC pipeline 

In [7]:
from sklearn.decomposition import PCA, FastICA
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from scipy.ndimage import gaussian_filter1d
    _HAS_SCI = True
except Exception:
    _HAS_SCI = False


def run_tSC_pipeline(phasic_time_signatures, tonic_time_signatures,
                     n_pca=5, freq_vec=np.arange(15,141,1),
                     zscore_features=True, mad_k=2.0,
                     weighted_alpha=0.4,
                     ignore_edge_bins=1):
    """
    Returns cycles_df with:
      - mode_freq_hz_*
      - *_strong versions (winner-strong for the mode_* fields, as before)
      - per-component strong flags (paper-style; overlaps allowed)
      - mode_freq_hz_featZw (feature-Z * |W_k|)
      - mode_freq_hz_proj   (within-cycle Z * |W_k|)
    Note: tSC strengths are the paper-faithful frequency-space projections (Xz @ W_freq.T).
    """

    # ---------------- 1) flatten ----------------
    def _flatten_sig_list(sig_list, label):
        rows, meta = [], []
        for i, arr in enumerate(sig_list):
            if not isinstance(arr, np.ndarray) or arr.size == 0:
                continue
            mask = np.isfinite(arr).all(axis=1)
            Xi = arr[mask]
            if Xi.size == 0:
                continue
            rows.append(Xi)
            for j in range(Xi.shape[0]):
                meta.append({"interval_idx": i, "cycle_idx_in_interval": j, "cycle_type": label})
        if len(rows) == 0:
            return np.zeros((0, len(freq_vec))), []
        return np.vstack(rows), meta

    X_phasic, meta_phasic = _flatten_sig_list(phasic_time_signatures, "phasic")
    X_tonic,  meta_tonic  = _flatten_sig_list(tonic_time_signatures, "tonic")
    X = np.vstack([X_phasic, X_tonic]); meta = meta_phasic + meta_tonic
    if X.shape[0] == 0:
        raise RuntimeError("No valid cycles to analyze.")

    # ---------------- 2) feature z-score ----------------
    if zscore_features:
        mu = X.mean(axis=0, keepdims=True)
        sd = X.std(axis=0, ddof=1, keepdims=True) + 1e-12
        Xz = (X - mu) / sd
    else:
        Xz = X

    # ---- smoothing helper ----
    def _smooth_rows(mat, sigma_hz, mode="reflect"):
        if _HAS_SCI:
            return gaussian_filter1d(mat, sigma=float(sigma_hz), axis=1, mode=mode)
        sigma = float(sigma_hz)
        rad = int(np.ceil(4 * sigma))
        kx = np.arange(-rad, rad + 1)
        ker = np.exp(-(kx**2) / (2 * sigma**2))
        ker /= ker.sum()
        pad = rad
        out = np.empty_like(mat)
        for i in range(mat.shape[0]):
            row = mat[i]
            if mode == "reflect":
                row_pad = np.r_[row[pad:0:-1], row, row[-2:-pad-2:-1]]
            elif mode == "constant":
                row_pad = np.r_[np.zeros(pad), row, np.zeros(pad)]
            else:
                row_pad = np.r_[row[pad:0:-1], row, row[-2:-pad-2:-1]]
            out[i] = np.convolve(row_pad, ker, mode="same")[pad:-pad]
        return out

    # ---------------- 2b) smoothed versions ----------------
    Xz_smooth5_ref  = _smooth_rows(Xz,  5.0, mode="reflect")
    Xz_smooth10_ref = _smooth_rows(Xz, 10.0, mode="reflect")
    Xz_smooth5_pad  = _smooth_rows(Xz,  5.0, mode="constant")
    Xz_smooth10_pad = _smooth_rows(Xz, 10.0, mode="constant")

    # ---------------- 3) PCA → ICA ----------------
    pca = PCA(n_components=n_pca, random_state=42)
    #Xz = Xz_smooth5_pad
    Z = pca.fit_transform(Xz_smooth5_pad)
    ica = FastICA(n_components=n_pca, random_state=42, max_iter=1000, tol=1e-3)
    S_latent  = ica.fit_transform(Z)            # latent sources (not used for strength anymore)
    W_freq = ica.components_ @ pca.components_ # components in frequency space (freq weights)

    # ---------------- 4) sign fix ----------------
    # keep your existing sign rule; it just needs to be applied to both S_latent and W_freq
    mean_src = S_latent.mean(axis=0, keepdims=True)
    signs = np.sign(mean_src); signs[signs == 0] = 1
    S_latent *= signs; W_freq *= signs.T

    # ---------------- 5) PAPER-FAITHFUL STRENGTHS ----------------
    strengths = Xz_smooth5_pad @ W_freq.T                        # (n_cycles, n_components)
    abs_strengths = np.abs(strengths)

    # thresholds & labels now computed from projection strengths
    def _mad_threshold(x, k=2.0):
        med = np.median(x); mad = np.median(np.abs(x - med)) + 1e-12
        return med + k * (mad / 0.6745)

    thr_per_comp = np.array([_mad_threshold(abs_strengths[:, k], k=mad_k) for k in range(n_pca)])
    labels_0based = np.argmax(abs_strengths, axis=1)

    # ========== winner-strong (your previous global mask; keep for continuity) ==========
    strong_mask = np.zeros_like(labels_0based, dtype=bool)
    for k in range(n_pca):
        strong_mask |= (labels_0based == k) & (abs_strengths[:, k] >= thr_per_comp[k])

    # ========== PAPER-STYLE per-component strong (OVERLAPS ALLOWED)  ==========
    # === NEW === boolean matrix [n_cycles, n_pca]
    strong_mask_matrix = abs_strengths >= thr_per_comp[None, :]

    # === NEW === handy summaries per cycle
    tsc_n_strong = strong_mask_matrix.sum(axis=1)                        # how many components are strong
    tsc_any_strong = tsc_n_strong > 0                                    # any strong?
    # list of 1-based component indices that are strong (e.g., [1, 3])
    tsc_strong_components = [list(np.nonzero(row)[0] + 1) for row in strong_mask_matrix]
    # string version for easy grouping/plotting ("", "2", "1,3", etc.)
    tsc_strong_components_str = [",".join(map(str, lst)) if len(lst) else "" for lst in tsc_strong_components]
    # exclusive label if exactly one strong; otherwise NaN
    tsc_exclusive_label = np.where(tsc_n_strong == 1,
                                   (np.argmax(strong_mask_matrix, axis=1) + 1),
                                   np.nan)

    # ---------------- helper: mode extraction ----------------
    def _mode_from_mat(mat):
        L, R = ignore_edge_bins, (mat.shape[1] - ignore_edge_bins)
        if R <= L:
            idx = np.argmax(mat, axis=1)
        else:
            idx = np.argmax(mat[:, L:R], axis=1) + L
        return freq_vec[idx]

    # ---------------- A) modes ----------------
    mode_freq_hz_featZ          = _mode_from_mat(Xz)
    mode_freq_hz_featZ_smooth   = _mode_from_mat(Xz_smooth5_ref)
    mode_freq_hz_featZ_smooth10 = _mode_from_mat(Xz_smooth10_ref)
    mode_freq_hz_featZ_smooth5_pad  = _mode_from_mat(Xz_smooth5_pad)
    mode_freq_hz_featZ_smooth10_pad = _mode_from_mat(Xz_smooth10_pad)

    # -------- A_strong: restrict to winner-strong cycles (as before) --------
    def _mode_strong(mat):
        modes = np.full(mat.shape[0], np.nan)
        modes[strong_mask] = _mode_from_mat(mat[strong_mask])
        return modes

    mode_freq_hz_featZ_strong          = _mode_strong(Xz)
    mode_freq_hz_featZ_smooth_strong   = _mode_strong(Xz_smooth5_pad)
    mode_freq_hz_featZ_smooth10_strong = _mode_strong(Xz_smooth10_pad)

    # ---------------- B) weighted modes (uses |W_k| + winner labels) ----------------
    def mode_from_feature_z_weighted(Xz_like, W_freq, labels_0based, freq_vec,
                                     avoid_edge_bins=1, alpha=None):
        n_cycles, n_freq = Xz_like.shape
        lo = avoid_edge_bins; hi = n_freq - avoid_edge_bins
        use_slice = slice(lo, hi) if hi > lo else slice(0, n_freq)
        modes = np.empty(n_cycles, dtype=float)
        for i in range(n_cycles):
            k = int(labels_0based[i])
            w = np.abs(W_freq[k]).copy()
            if alpha is not None:
                thr = alpha * np.max(w)
                m = (w >= thr)
                if m.sum() >= 3:
                    w[~m] = 0.0
            y = Xz_like[i] * w
            yseg = y[use_slice]
            idx_rel = int(np.argmax(yseg))
            idx = (lo + idx_rel) if hi > lo else idx_rel
            modes[i] = freq_vec[idx]
        return modes

    mode_freq_hz_featZ_weighted = mode_from_feature_z_weighted(
        Xz, W_freq, labels_0based, freq_vec,
        avoid_edge_bins=ignore_edge_bins, alpha=weighted_alpha
    )

    # ---------------- C) within-cycle-Z + weights ----------------
    eps = 1e-12
    def tsc_weighted_mode_freq(X, W_freq, labels_0based, freq_vec,
                               avoid_edge_bins=1, alpha=0.4):
        n_cycles, n_freq = X.shape
        modes = np.empty(n_cycles, dtype=float)
        lo = avoid_edge_bins; hi = n_freq - avoid_edge_bins
        use_slice = slice(lo, hi) if hi > lo else slice(0, n_freq)
        for i in range(n_cycles):
            k = int(labels_0based[i])
            x = X[i]
            xz = (x - x.mean()) / (x.std(ddof=1) + eps)
            w = np.abs(W_freq[k]).copy()
            if alpha is not None:
                thr = alpha * np.max(w)
                mask = w >= thr
                if mask.sum() >= 3:
                    w[~mask] = 0.0
            y = xz * w
            yseg = y[use_slice]
            idx_rel = int(np.argmax(yseg))
            idx = (lo + idx_rel) if hi > lo else idx_rel
            modes[i] = freq_vec[idx]
        return modes

    mode_freq_hz_proj = tsc_weighted_mode_freq(
        X, W_freq, labels_0based, freq_vec,
        avoid_edge_bins=ignore_edge_bins, alpha=weighted_alpha
    )

    # ---------------- 7) package ----------------
    cycles_df = pd.DataFrame(meta)
    cycles_df["tSC_label"]                     = labels_0based + 1
    cycles_df["mode_freq_hz_featZ"]            = mode_freq_hz_featZ
    cycles_df["mode_freq_hz_featZ_smooth"]     = mode_freq_hz_featZ_smooth
    cycles_df["mode_freq_hz_featZ_smooth10"]   = mode_freq_hz_featZ_smooth10
    cycles_df["mode_freq_hz_featZ_smooth5_pad"]= mode_freq_hz_featZ_smooth5_pad
    cycles_df["mode_freq_hz_featZ_smooth10_pad"]= mode_freq_hz_featZ_smooth10_pad
    cycles_df["mode_freq_hz_featZ_strong"]     = mode_freq_hz_featZ_strong
    cycles_df["mode_freq_hz_featZ_smooth_strong"]   = mode_freq_hz_featZ_smooth_strong
    cycles_df["mode_freq_hz_featZ_smooth10_strong"] = mode_freq_hz_featZ_smooth10_strong
    cycles_df["mode_freq_hz_featZw"]           = mode_freq_hz_featZ_weighted
    cycles_df["mode_freq_hz_proj"]             = mode_freq_hz_proj

    # per-component strengths/flags
    for k in range(n_pca):
        cycles_df[f"tSC{k+1}_strength"] = strengths[:, k]
        cycles_df[f"tSC{k+1}_strong"]   = (abs_strengths[:, k] >= thr_per_comp[k])

    # === NEW === paper-style summaries (overlaps allowed)
    cycles_df["tSC_any_strong"]           = tsc_any_strong
    cycles_df["tSC_n_strong"]             = tsc_n_strong
    cycles_df["tSC_strong_components"]    = tsc_strong_components            # list[ints]
    cycles_df["tSC_strong_components_str"]= tsc_strong_components_str        # "1,3" etc.
    cycles_df["tSC_winner_strong"]        = strong_mask                      # winner-only strong
    cycles_df["tSC_exclusive_label"]      = tsc_exclusive_label              # 1-based or NaN

    tSC_results = {
        "freq_vec": freq_vec,
        "pca": pca,
        "ica": ica,
        "weights_freq": W_freq,                 # freq-space weight vectors (for plotting components)
        "strengths": strengths,                 # projection strengths used for labels/thresholds
        "latent_sources_S": S_latent,           # optional: latent ICA sources
        "thresholds_abs_strength": thr_per_comp,
        "X_cycles": X,
        "X_cycles_featZ": Xz,
        "X_cycles_featZ_smooth":    Xz_smooth5_ref,
        "X_cycles_featZ_smooth10":  Xz_smooth10_ref,
        "X_cycles_featZ_smooth5_pad":  Xz_smooth5_pad,
        "X_cycles_featZ_smooth10_pad": Xz_smooth10_pad,
        # === NEW === expose masks for downstream analyses/plots
        "strong_mask_matrix": strong_mask_matrix,   # shape (n_cycles, n_pca) paper-style overlaps
        "winner_strong_mask": strong_mask,          # shape (n_cycles,) winner-only
        "meta": meta
    }

    print("=== PCA/ICA done ===")
    print("n cycles:", X.shape[0])
    print("Mode — featZ (median):", np.nanmedian(cycles_df['mode_freq_hz_featZ']))
    print("Mode — featZ strong (median):", np.nanmedian(cycles_df['mode_freq_hz_featZ_strong']))
    print("Mode — featZ 5Hz reflect strong (median):", np.nanmedian(cycles_df['mode_freq_hz_featZ_smooth_strong']))
    print("Mode — featZ 10Hz reflect strong (median):", np.nanmedian(cycles_df['mode_freq_hz_featZ_smooth10_strong']))
    return cycles_df, tSC_results

In [8]:
cycles_df, tSC_results = run_tSC_pipeline(phasic_time_signatures, tonic_time_signatures,
                                          n_pca=5, freq_vec=np.arange(15,141,1),
                                          zscore_features=False, mad_k=2.0)

=== PCA/ICA done ===
n cycles: 51844
Mode — featZ (median): 34.0
Mode — featZ strong (median): 38.0
Mode — featZ 5Hz reflect strong (median): 55.0
Mode — featZ 10Hz reflect strong (median): 64.0


In [9]:
# --- imports you already have ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import umap  # pip install umap-learn

META_COLS = ['rat_id','condition','trial','cycle_type','SD','date']

# --------- 1) Make block list from waveforms_df (in its actual row order) ----------
def _blocks_from_waveforms(waveforms_df, by_cols=('condition','trial','cycle_type','SD','date')):
    if not all(c in waveforms_df.columns for c in by_cols):
        raise ValueError(f"waveforms_df is missing required meta columns: {by_cols}")
    # identify where any meta value changes → start a new block
    meta = waveforms_df[list(by_cols)].astype(str).to_numpy()
    # first row starts a block
    is_new = np.zeros(len(waveforms_df), dtype=bool)
    is_new[0] = True
    is_new[1:] = np.any(meta[1:] != meta[:-1], axis=1)

    block_ids = np.cumsum(is_new) - 1
    waveforms_df = waveforms_df.copy()
    waveforms_df["_block_id"] = block_ids

    blocks = []
    for bid, grp in waveforms_df.groupby("_block_id", sort=False):
        # store block metadata and size
        row0 = grp.iloc[0]
        blocks.append({
            "block_id": int(bid),
            "cycle_type": str(row0["cycle_type"]),
            "condition":  row0["condition"],
            "trial":      row0["trial"],
            "SD":         row0["SD"],
            "date":       row0["date"],
            "size":       len(grp)
        })
    return blocks

# --------- 2) Reorder cycles_df to match those blocks ----------
def reorder_cycles_like_waveforms(waveforms_df, cycles_df):
    if len(waveforms_df) != len(cycles_df):
        raise ValueError(f"Size mismatch: waveforms_df={len(waveforms_df)}, cycles_df={len(cycles_df)}")

    blocks = _blocks_from_waveforms(waveforms_df)

    # partition cycles_df into phasic and tonic queues
    if "cycle_type" not in cycles_df.columns:
        raise ValueError("cycles_df must contain a 'cycle_type' column (from run_tSC_pipeline meta).")

    cyc_ph = cycles_df.loc[cycles_df["cycle_type"].astype(str)=="phasic"].reset_index(drop=True)
    cyc_to = cycles_df.loc[cycles_df["cycle_type"].astype(str)=="tonic" ].reset_index(drop=True)
    ip = it = 0

    parts = []
    for b in blocks:
        n = int(b["size"])
        if b["cycle_type"] == "phasic":
            part = cyc_ph.iloc[ip:ip+n]
            if len(part) != n:
                raise ValueError(f"Not enough PHASIC rows to fill block {b}. Needed {n}, had {len(cyc_ph)-ip}.")
            ip += n
        else:
            part = cyc_to.iloc[it:it+n]
            if len(part) != n:
                raise ValueError(f"Not enough TONIC rows to fill block {b}. Needed {n}, had {len(cyc_to)-it}.")
            it += n
        parts.append(part)

    cycles_reordered = pd.concat(parts, ignore_index=True)
    assert len(cycles_reordered) == len(cycles_df)
    return cycles_reordered

# --------- 3) Build UMAP features (waveforms ± mode scalar) ----------
def _waveform_feature_matrix(waveforms_df):
    cols = [c for c in waveforms_df.columns if c not in META_COLS]
    return waveforms_df[cols].to_numpy(), cols

def build_umap_features(waveforms_df, cycles_df_aligned,
                        include_mode_in_fit=True,
                        mode_col='mode_freq_hz_featZ_smooth',
                        standardize='z'):
    Xw, _ = _waveform_feature_matrix(waveforms_df)
    if not include_mode_in_fit:
        return Xw, cycles_df_aligned[mode_col].to_numpy()

    if mode_col not in cycles_df_aligned.columns:
        raise KeyError(f"'{mode_col}' not found in cycles_df_aligned columns.")

    mode = cycles_df_aligned[mode_col].to_numpy().reshape(-1, 1).astype(float)
    if standardize == 'z':
        m = np.nanmean(mode, axis=0); s = np.nanstd(mode, axis=0) + 1e-12
        mode_std = (mode - m) / s
    elif standardize == 'minmax':
        mn = np.nanmin(mode, axis=0); mx = np.nanmax(mode, axis=0)
        mode_std = (mode - mn) / (mx - mn + 1e-12)
    else:
        mode_std = mode

    return np.hstack([Xw, mode_std]), mode.ravel()

# --------- 4) Fit & plot ----------
def umap_color_by_mode(waveforms_df, cycles_df,
                       include_mode_in_fit=True,
                       n_neighbors=15, min_dist=0.1, n_components=2,
                       random_state=42):
    # hard alignment using real block order
    cy_aligned = reorder_cycles_like_waveforms(waveforms_df, cycles_df)

    X_umap, mode_vals = build_umap_features(
        waveforms_df, cy_aligned,
        include_mode_in_fit=include_mode_in_fit,
        mode_col='mode_freq_hz_featZ_smooth',
        standardize='z'
    )

    # NOTE: if you want parallelism without the "n_jobs overridden" warning, set random_state=None
    rs = random_state
    um = umap.UMAP(n_neighbors=n_neighbors, min_dist=min_dist,
                   n_components=n_components, metric='euclidean',
                   random_state=rs)
    emb = um.fit_transform(X_umap)

    vmin = np.nanpercentile(mode_vals, 1)
    vmax = np.nanpercentile(mode_vals, 99)
    plt.figure(figsize=(6,5))
    plt.scatter(emb[:,0], emb[:,1], c=mode_vals, vmin=vmin, vmax=vmax, cmap='hot', s=12)
    plt.colorbar(label='mode_freq_hz_featZ_smooth')
    ttl = 'UMAP: waveforms + mode' if include_mode_in_fit else 'UMAP: waveforms only (colored by mode)'
    plt.title(ttl)
    plt.tight_layout()
    return emb, cy_aligned, mode_vals

In [10]:
emb_with_mode, cycles_df_aligned, mode_vals = umap_color_by_mode(
    waveforms_df, cycles_df,
    include_mode_in_fit=True,
    n_neighbors=15, min_dist=0.1, n_components=2, random_state=42
)

ValueError: Size mismatch: waveforms_df=44861, cycles_df=51844

In [9]:
def plot_umap_one_rat_waveforms(
    waveforms_df: pd.DataFrame,
    cycles_df: pd.DataFrame,
    trials_df: pd.DataFrame = None,                 # optional; if provided we align by keys
    mode_col: str = "mode_freq_hz_featZ_smooth",    # used for color AND included as a feature
    scale_features: bool = True,
    n_neighbors: int = 15,
    min_dist: float = 0.1,
    random_state: int = 42,
):
    """
    Single-rat UMAP of waveform features (+ the selected mode column).
    Color encodes the same mode column values (Hz). Phasic vs Tonic are marked differently.

    Inputs:
      - waveforms_df : DataFrame with cycle-level waveform features for ONE rat
      - cycles_df    : DataFrame returned by run_tSC_pipeline (same rat)
      - trials_df    : (optional) trial/cycle metadata that shares alignment keys with cycles_df
      - mode_col     : a column in cycles_df (e.g., 'mode_freq_hz_featZ_smooth' or '*_smooth5_pad')

    Notes:
      • If trials_df is provided and both it and cycles_df have keys
        ['interval_idx','cycle_idx_in_interval','cycle_type'], we merge by those keys.
      • Otherwise we fall back to stable order matching within each cycle_type.
    """
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from sklearn.preprocessing import StandardScaler

    # ---------- 0) Basic checks ----------
    if not isinstance(waveforms_df, pd.DataFrame):
        raise RuntimeError("waveforms_df must be a DataFrame.")
    if not isinstance(cycles_df, pd.DataFrame):
        raise RuntimeError("cycles_df must be a DataFrame.")

    # pick a usable mode column if the requested one is missing
    if mode_col not in cycles_df.columns:
        alt_cols = [c for c in cycles_df.columns if c.startswith("mode_freq_hz_")]
        if "mode_freq_hz_featZ_smooth5_pad" in alt_cols:
            mode_col = "mode_freq_hz_featZ_smooth5_pad"
        elif len(alt_cols):
            mode_col = alt_cols[0]
        else:
            raise RuntimeError("No 'mode_freq_hz_*' columns found in cycles_df.")

    # unify types / casing
    def _coerce(df):
        out = df.copy()
        if "cycle_type" in out.columns:
            out["cycle_type"] = out["cycle_type"].astype(str).str.lower()
        return out

    wave = _coerce(waveforms_df)
    cyc   = _coerce(cycles_df)

    # ---------- 1) Attach the mode column to waveforms rows ----------
    # Preferred alignment: via trials_df (exact keys)
    merged = None
    align_keys = ["interval_idx", "cycle_idx_in_interval", "cycle_type"]
    if trials_df is not None:
        tri = _coerce(trials_df)
        if all(k in tri.columns for k in align_keys) and all(k in cyc.columns for k in align_keys):
            # pull only the two needed cols from cycles_df: keys + mode_col
            cyc_small = cyc[align_keys + [mode_col]].copy()
            merged = tri.merge(cyc_small, on=align_keys, how="left", validate="one_to_one")
            # merge this onto waveforms_df by the same keys if they exist there
            if all(k in wave.columns for k in align_keys):
                merged = wave.merge(merged[align_keys + [mode_col]], on=align_keys, how="left")
            else:
                # if waveforms_df lacks keys, assume 1:1 order with trials_df
                if len(wave) != len(merged):
                    raise RuntimeError(
                        "Cannot align waveforms_df with trials_df: key columns missing in waveforms_df "
                        "and lengths differ. Please provide alignment keys."
                    )
                merged = wave.copy()
                merged[mode_col] = merged.index.map(merged.index.to_series().map(merged := merged))  # noop to keep name
                merged[mode_col] = cyc_small[mode_col].to_numpy()  # use cycles order after tri merge
        # if trials_df provided but missing keys, we’ll use fallback below

    # Fallback alignment: stable order within each cycle_type
    if merged is None:
        def _group_order_index(df: pd.DataFrame, keys):
            g = df[keys].copy().fillna("__NA__")
            return g.groupby(keys).cumcount()

        # create stable order index per type (phasic/tonic)
        if "cycle_type" not in wave.columns or "cycle_type" not in cyc.columns:
            raise RuntimeError("Need 'cycle_type' in both dataframes for fallback alignment.")

        wave = wave.copy()
        cyc  = cyc.copy()
        wave["gidx"] = _group_order_index(wave, ["cycle_type"])
        cyc["gidx"]  = _group_order_index(cyc,  ["cycle_type"])

        merged = wave.merge(
            cyc[["cycle_type", "gidx", mode_col]],
            on=["cycle_type", "gidx"],
            how="inner"
        )

    if merged.empty:
        raise RuntimeError("Alignment produced an empty DataFrame.")

    # ---------- 2) Build feature matrix (include the chosen mode_col) ----------
    numeric_cols = merged.select_dtypes(include=[np.number]).columns.tolist()

    # Exclude obvious metadata columns and *other* tSC/mode columns
    meta_cols = {
        "condition","trial","cycle_type","SD","date",
        "interval_idx","cycle_idx_in_interval","gidx"
    }
    meta_cols |= {c for c in merged.columns if c.startswith("tSC")}
    meta_cols |= {c for c in merged.columns if (c.startswith("mode_freq_hz_") and c != mode_col)}

    feature_cols = [c for c in numeric_cols if c not in meta_cols]
    # ensure our selected mode column is included (as feature)
    if mode_col not in feature_cols:
        merged[mode_col] = pd.to_numeric(merged[mode_col], errors="coerce")
        feature_cols.append(mode_col)

    if not feature_cols:
        raise RuntimeError("No numeric feature columns found for embedding.")

    X = merged[feature_cols].to_numpy()
    color = pd.to_numeric(merged[mode_col], errors="coerce").to_numpy().astype(float)

    valid = np.isfinite(color) & np.isfinite(X).all(axis=1)
    merged = merged.loc[valid].reset_index(drop=True)
    X = X[valid]
    color = color[valid]

    # ---------- 3) Scale (optional) ----------
    if scale_features:
        X = StandardScaler().fit_transform(X)

    # ---------- 4) Compute embedding (UMAP → t-SNE fallback) ----------
    try:
        import umap
        reducer = umap.UMAP(
            n_neighbors=n_neighbors, min_dist=min_dist,
            n_components=2, metric="euclidean", random_state=random_state
        )
        emb = reducer.fit_transform(X)
        xlab, ylab, method = "UMAP-1", "UMAP-2", "UMAP"
    except Exception:
        from sklearn.manifold import TSNE
        tsne = TSNE(n_components=2, perplexity=30, learning_rate="auto",
                    init="pca", random_state=random_state)
        emb = tsne.fit_transform(X)
        xlab, ylab, method = "Dim 1", "Dim 2", "t-SNE"

    # ---------- 5) Plot ----------
    if "cycle_type" not in merged.columns:
        raise RuntimeError("Missing 'cycle_type' after alignment.")

    phasic_m = (merged["cycle_type"].values == "phasic")
    tonic_m  = (merged["cycle_type"].values == "tonic")

    vmin = np.nanpercentile(color, 1)
    vmax = np.nanpercentile(color, 99)

    plt.figure(figsize=(10, 6))
    sc1 = plt.scatter(
        emb[tonic_m, 0], emb[tonic_m, 1],
        c=color[tonic_m], cmap="hot", vmin=vmin, vmax=vmax,
        s=28, alpha=0.80, edgecolors="none", label="Tonic"
    )
    plt.scatter(
        emb[phasic_m, 0], emb[phasic_m, 1],
        c=color[phasic_m], cmap="hot", vmin=vmin, vmax=vmax,
        s=42, marker="X", edgecolors="k", linewidths=0.25, label="Phasic"
    )
    plt.xlabel(xlab); plt.ylabel(ylab)
    plt.title(f"{method} of waveform features (+ {mode_col}) — color: {mode_col} (Hz)")
    cbar = plt.colorbar(sc1); cbar.set_label(f"{mode_col} (Hz)")
    plt.legend(frameon=False)
    plt.tight_layout()
    plt.show()


cycles_df, tSC_results = run_tSC_pipeline(
    phasic_time_signatures, tonic_time_signatures,
    n_pca=5, freq_vec=np.arange(15,141,1),
    zscore_features=False, mad_k=2.0
)

# Then call the plotter (it will use trials_df for key-based alignment if provided)
plot_umap_one_rat_waveforms(
    waveforms_df=waveforms_df,
    cycles_df=cycles_df,
    trials_df=trials_df,                        # pass None if you want to use fallback alignment
    mode_col="mode_freq_hz_featZ",       # or "mode_freq_hz_featZ_smooth5_pad"
    scale_features=True,
    n_neighbors=15,
    min_dist=0.1,
    random_state=42,
)

NameError: name 'phasic_time_signatures' is not defined

In [8]:
import numpy as np
import pandas as pd
import pickle
from pathlib import Path
from sklearn.decomposition import PCA, FastICA

try:
    from scipy.ndimage import gaussian_filter1d
    _HAS_SCI = True
except Exception:
    _HAS_SCI = False


def load_rat_data(rid, out_root=Path("./fppsig_v3_outputs")):
    """Load all data objects for a single rat."""
    rat_dir = out_root / f"Rat{rid}"
    
    # Option 1: combined pickle
    all_pkl = rat_dir / "all_objects.pkl"
    if all_pkl.exists():
        with open(all_pkl, "rb") as f:
            return pickle.load(f)
    
    # Option 2: separate files
    def _load_df(name):
        for ext in [".parquet", ".csv"]:
            p = rat_dir / f"{name}{ext}"
            if p.exists():
                return pd.read_parquet(p) if ext == ".parquet" else pd.read_csv(p)
        return None
    
    waveforms_df = _load_df("waveforms_df")
    trials_df = _load_df("trials_df")
    
    lists_pkl = rat_dir / "lists.pkl"
    if lists_pkl.exists():
        with open(lists_pkl, "rb") as f:
            lists = pickle.load(f)
    else:
        lists = {}
    
    return dict(waveforms_df=waveforms_df, trials_df=trials_df, **lists)


def load_all_rats(out_root=Path("./fppsig_v3_outputs")):
    """Load all rats listed in manifest.csv into a dict."""
    manifest_path = out_root / "manifest.csv"
    if not manifest_path.exists():
        raise FileNotFoundError(f"Manifest not found at {manifest_path}")
    
    manifest = pd.read_csv(manifest_path)
    rat_ids = manifest["rat_id"].tolist()
    
    all_data = {}
    for rid in rat_ids:
        print(f"Loading Rat {rid} ...", end=" ")
        try:
            all_data[rid] = load_rat_data(rid, out_root)
            print("✅ done")
        except Exception as e:
            print(f"⚠️ failed ({e})")
    
    return all_data


def run_multi_rat_tSC_pipeline(all_rats_data, 
                                n_pca=5, 
                                freq_vec=np.arange(15, 141, 1),
                                zscore_features=True, 
                                mad_k=2.0,
                                weighted_alpha=0.4,
                                ignore_edge_bins=1):
    """
    Run tSC pipeline on combined data from all rats.
    
    Parameters:
    -----------
    all_rats_data : dict
        Dictionary with rat_id as keys, each containing:
        - phasic_time_signatures: list of arrays
        - tonic_time_signatures: list of arrays
    
    Returns:
    --------
    cycles_df : pd.DataFrame
        DataFrame with all cycles from all rats, including rat_id and all tSC metrics
    tSC_results : dict
        Dictionary containing PCA/ICA components and other analysis results
    """
    
    # ============ 1) Combine data from all rats ============
    print("Combining data from all rats...")
    
    all_phasic_sigs = []
    all_tonic_sigs = []
    rat_metadata = []  # track which rat each signature belongs to
    
    for rat_id, data in all_rats_data.items():
        phasic_sigs = data.get('phasic_time_signatures', [])
        tonic_sigs = data.get('tonic_time_signatures', [])
        
        # Track rat_id for each segment
        for seg in phasic_sigs:
            if isinstance(seg, np.ndarray) and seg.size > 0:
                all_phasic_sigs.append(seg)
                rat_metadata.append({'rat_id': rat_id, 'cycle_type': 'phasic'})
        
        for seg in tonic_sigs:
            if isinstance(seg, np.ndarray) and seg.size > 0:
                all_tonic_sigs.append(seg)
                rat_metadata.append({'rat_id': rat_id, 'cycle_type': 'tonic'})
    
    print(f"  Total phasic segments: {len(all_phasic_sigs)}")
    print(f"  Total tonic segments: {len(all_tonic_sigs)}")
    
    # ============ 2) Flatten all signatures ============
    def _flatten_sig_list(sig_list, metadata_list, label):
        rows, meta = [], []
        seg_idx = 0
        
        for sig_group in sig_list:
            # Find corresponding metadata
            rat_meta = [m for m in metadata_list if m['cycle_type'] == label][seg_idx] if seg_idx < len([m for m in metadata_list if m['cycle_type'] == label]) else {}
            
            if not isinstance(sig_group, np.ndarray) or sig_group.size == 0:
                continue
            
            mask = np.isfinite(sig_group).all(axis=1)
            Xi = sig_group[mask]
            
            if Xi.size == 0:
                continue
            
            rows.append(Xi)
            for j in range(Xi.shape[0]):
                meta.append({
                    "interval_idx": seg_idx,
                    "cycle_idx_in_interval": j,
                    "cycle_type": label,
                    "rat_id": rat_meta.get('rat_id', None)
                })
            
            seg_idx += 1
        
        if len(rows) == 0:
            return np.zeros((0, len(freq_vec))), []
        
        return np.vstack(rows), meta
    
    # Separate metadata by cycle type
    phasic_meta = [m for m in rat_metadata if m['cycle_type'] == 'phasic']
    tonic_meta = [m for m in rat_metadata if m['cycle_type'] == 'tonic']
    
    X_phasic, meta_phasic = _flatten_sig_list(all_phasic_sigs, phasic_meta, "phasic")
    X_tonic, meta_tonic = _flatten_sig_list(all_tonic_sigs, tonic_meta, "tonic")
    
    X = np.vstack([X_phasic, X_tonic])
    meta = meta_phasic + meta_tonic
    
    if X.shape[0] == 0:
        raise RuntimeError("No valid cycles to analyze.")
    
    print(f"Total cycles combined: {X.shape[0]}")
    print(f"  Phasic: {X_phasic.shape[0]}")
    print(f"  Tonic: {X_tonic.shape[0]}")
    
    # ============ 3) Feature z-score ============
    if zscore_features:
        mu = X.mean(axis=0, keepdims=True)
        sd = X.std(axis=0, ddof=1, keepdims=True) + 1e-12
        Xz = (X - mu) / sd
    else:
        Xz = X
    
    # ============ 4) Smoothing helper ============
    def _smooth_rows(mat, sigma_hz, mode="reflect"):
        if _HAS_SCI:
            return gaussian_filter1d(mat, sigma=float(sigma_hz), axis=1, mode=mode)
        
        sigma = float(sigma_hz)
        rad = int(np.ceil(4 * sigma))
        kx = np.arange(-rad, rad + 1)
        ker = np.exp(-(kx**2) / (2 * sigma**2))
        ker /= ker.sum()
        pad = rad
        out = np.empty_like(mat)
        
        for i in range(mat.shape[0]):
            row = mat[i]
            if mode == "reflect":
                row_pad = np.r_[row[pad:0:-1], row, row[-2:-pad-2:-1]]
            elif mode == "constant":
                row_pad = np.r_[np.zeros(pad), row, np.zeros(pad)]
            else:
                row_pad = np.r_[row[pad:0:-1], row, row[-2:-pad-2:-1]]
            out[i] = np.convolve(row_pad, ker, mode="same")[pad:-pad]
        
        return out
    
    # ============ 5) Create smoothed versions ============
    Xz_smooth5_ref = _smooth_rows(Xz, 5.0, mode="reflect")
    Xz_smooth10_ref = _smooth_rows(Xz, 10.0, mode="reflect")
    Xz_smooth5_pad = _smooth_rows(Xz, 5.0, mode="constant")
    Xz_smooth10_pad = _smooth_rows(Xz, 10.0, mode="constant")
    
    # ============ 6) PCA → ICA ============
    print("Running PCA...")
    pca = PCA(n_components=n_pca, random_state=42)
    Z = pca.fit_transform(Xz)
    
    print("Running ICA...")
    ica = FastICA(n_components=n_pca, random_state=42, max_iter=1000, tol=1e-3)
    S_latent = ica.fit_transform(Z)
    W_freq = ica.components_ @ pca.components_
    
    # ============ 7) Sign fix ============
    mean_src = S_latent.mean(axis=0, keepdims=True)
    signs = np.sign(mean_src)
    signs[signs == 0] = 1
    S_latent *= signs
    W_freq *= signs.T
    
    # ============ 8) Compute strengths ============
    strengths = Xz @ W_freq.T
    abs_strengths = np.abs(strengths)
    
    # ============ 9) Thresholds ============
    def _mad_threshold(x, k=2.0):
        med = np.median(x)
        mad = np.median(np.abs(x - med)) + 1e-12
        return med + k * (mad / 0.6745)
    
    thr_per_comp = np.array([_mad_threshold(abs_strengths[:, k], k=mad_k) 
                              for k in range(n_pca)])
    
    labels_0based = np.argmax(abs_strengths, axis=1)
    
    # ============ 10) Winner-strong mask ============
    strong_mask = np.zeros_like(labels_0based, dtype=bool)
    for k in range(n_pca):
        strong_mask |= (labels_0based == k) & (abs_strengths[:, k] >= thr_per_comp[k])
    
    # ============ 11) Paper-style per-component strong ============
    strong_mask_matrix = abs_strengths >= thr_per_comp[None, :]
    tsc_n_strong = strong_mask_matrix.sum(axis=1)
    tsc_any_strong = tsc_n_strong > 0
    tsc_strong_components = [list(np.nonzero(row)[0] + 1) for row in strong_mask_matrix]
    tsc_strong_components_str = [",".join(map(str, lst)) if len(lst) else "" 
                                  for lst in tsc_strong_components]
    tsc_exclusive_label = np.where(tsc_n_strong == 1,
                                    (np.argmax(strong_mask_matrix, axis=1) + 1),
                                    np.nan)
    
    # ============ 12) Mode extraction helper ============
    def _mode_from_mat(mat):
        L, R = ignore_edge_bins, (mat.shape[1] - ignore_edge_bins)
        if R <= L:
            idx = np.argmax(mat, axis=1)
        else:
            idx = np.argmax(mat[:, L:R], axis=1) + L
        return freq_vec[idx]
    
    # ============ 13) Extract modes ============
    mode_freq_hz_featZ = _mode_from_mat(Xz)
    mode_freq_hz_featZ_smooth = _mode_from_mat(Xz_smooth5_ref)
    mode_freq_hz_featZ_smooth10 = _mode_from_mat(Xz_smooth10_ref)
    mode_freq_hz_featZ_smooth5_pad = _mode_from_mat(Xz_smooth5_pad)
    mode_freq_hz_featZ_smooth10_pad = _mode_from_mat(Xz_smooth10_pad)
    
    # ============ 14) Strong modes ============
    def _mode_strong(mat):
        modes = np.full(mat.shape[0], np.nan)
        modes[strong_mask] = _mode_from_mat(mat[strong_mask])
        return modes
    
    mode_freq_hz_featZ_strong = _mode_strong(Xz)
    mode_freq_hz_featZ_smooth_strong = _mode_strong(Xz_smooth5_ref)
    mode_freq_hz_featZ_smooth10_strong = _mode_strong(Xz_smooth10_ref)
    
    # ============ 15) Weighted modes ============
    def mode_from_feature_z_weighted(Xz_like, W_freq, labels_0based, freq_vec,
                                      avoid_edge_bins=1, alpha=None):
        n_cycles, n_freq = Xz_like.shape
        lo = avoid_edge_bins
        hi = n_freq - avoid_edge_bins
        use_slice = slice(lo, hi) if hi > lo else slice(0, n_freq)
        modes = np.empty(n_cycles, dtype=float)
        
        for i in range(n_cycles):
            k = int(labels_0based[i])
            w = np.abs(W_freq[k]).copy()
            
            if alpha is not None:
                thr = alpha * np.max(w)
                m = (w >= thr)
                if m.sum() >= 3:
                    w[~m] = 0.0
            
            y = Xz_like[i] * w
            yseg = y[use_slice]
            idx_rel = int(np.argmax(yseg))
            idx = (lo + idx_rel) if hi > lo else idx_rel
            modes[i] = freq_vec[idx]
        
        return modes
    
    mode_freq_hz_featZ_weighted = mode_from_feature_z_weighted(
        Xz, W_freq, labels_0based, freq_vec,
        avoid_edge_bins=ignore_edge_bins, alpha=weighted_alpha
    )
    
    # ============ 16) Within-cycle-Z + weights ============
    eps = 1e-12
    
    def tsc_weighted_mode_freq(X, W_freq, labels_0based, freq_vec,
                                avoid_edge_bins=1, alpha=0.4):
        n_cycles, n_freq = X.shape
        modes = np.empty(n_cycles, dtype=float)
        lo = avoid_edge_bins
        hi = n_freq - avoid_edge_bins
        use_slice = slice(lo, hi) if hi > lo else slice(0, n_freq)
        
        for i in range(n_cycles):
            k = int(labels_0based[i])
            x = X[i]
            xz = (x - x.mean()) / (x.std(ddof=1) + eps)
            w = np.abs(W_freq[k]).copy()
            
            if alpha is not None:
                thr = alpha * np.max(w)
                mask = w >= thr
                if mask.sum() >= 3:
                    w[~mask] = 0.0
            
            y = xz * w
            yseg = y[use_slice]
            idx_rel = int(np.argmax(yseg))
            idx = (lo + idx_rel) if hi > lo else idx_rel
            modes[i] = freq_vec[idx]
        
        return modes
    
    mode_freq_hz_proj = tsc_weighted_mode_freq(
        X, W_freq, labels_0based, freq_vec,
        avoid_edge_bins=ignore_edge_bins, alpha=weighted_alpha
    )
    
    # ============ 17) Package results ============
    cycles_df = pd.DataFrame(meta)
    cycles_df["tSC_label"] = labels_0based + 1
    cycles_df["mode_freq_hz_featZ"] = mode_freq_hz_featZ
    cycles_df["mode_freq_hz_featZ_smooth"] = mode_freq_hz_featZ_smooth
    cycles_df["mode_freq_hz_featZ_smooth10"] = mode_freq_hz_featZ_smooth10
    cycles_df["mode_freq_hz_featZ_smooth5_pad"] = mode_freq_hz_featZ_smooth5_pad
    cycles_df["mode_freq_hz_featZ_smooth10_pad"] = mode_freq_hz_featZ_smooth10_pad
    cycles_df["mode_freq_hz_featZ_strong"] = mode_freq_hz_featZ_strong
    cycles_df["mode_freq_hz_featZ_smooth_strong"] = mode_freq_hz_featZ_smooth_strong
    cycles_df["mode_freq_hz_featZ_smooth10_strong"] = mode_freq_hz_featZ_smooth10_strong
    cycles_df["mode_freq_hz_featZw"] = mode_freq_hz_featZ_weighted
    cycles_df["mode_freq_hz_proj"] = mode_freq_hz_proj
    
    # Per-component strengths/flags
    for k in range(n_pca):
        cycles_df[f"tSC{k+1}_strength"] = strengths[:, k]
        cycles_df[f"tSC{k+1}_strong"] = (abs_strengths[:, k] >= thr_per_comp[k])
    
    # Paper-style summaries
    cycles_df["tSC_any_strong"] = tsc_any_strong
    cycles_df["tSC_n_strong"] = tsc_n_strong
    cycles_df["tSC_strong_components"] = tsc_strong_components
    cycles_df["tSC_strong_components_str"] = tsc_strong_components_str
    cycles_df["tSC_winner_strong"] = strong_mask
    cycles_df["tSC_exclusive_label"] = tsc_exclusive_label
    
    tSC_results = {
        "freq_vec": freq_vec,
        "pca": pca,
        "ica": ica,
        "weights_freq": W_freq,
        "strengths": strengths,
        "latent_sources_S": S_latent,
        "thresholds_abs_strength": thr_per_comp,
        "X_cycles": X,
        "X_cycles_featZ": Xz,
        "X_cycles_featZ_smooth": Xz_smooth5_ref,
        "X_cycles_featZ_smooth10": Xz_smooth10_ref,
        "X_cycles_featZ_smooth5_pad": Xz_smooth5_pad,
        "X_cycles_featZ_smooth10_pad": Xz_smooth10_pad,
        "strong_mask_matrix": strong_mask_matrix,
        "winner_strong_mask": strong_mask,
        "meta": meta
    }
    
    # ============ 18) Print summary ============
    print("\n=== Multi-Rat PCA/ICA Complete ===")
    print(f"Total cycles: {X.shape[0]}")
    print(f"Mode — featZ (median): {np.nanmedian(cycles_df['mode_freq_hz_featZ']):.2f} Hz")
    print(f"Mode — featZ strong (median): {np.nanmedian(cycles_df['mode_freq_hz_featZ_strong']):.2f} Hz")
    print(f"Mode — featZ 5Hz reflect strong (median): {np.nanmedian(cycles_df['mode_freq_hz_featZ_smooth_strong']):.2f} Hz")
    print(f"Mode — featZ 10Hz reflect strong (median): {np.nanmedian(cycles_df['mode_freq_hz_featZ_smooth10_strong']):.2f} Hz")
    
    # Per-rat summary
    print("\nPer-rat cycle counts:")
    rat_counts = cycles_df.groupby('rat_id').size()
    for rat_id, count in rat_counts.items():
        print(f"  Rat {rat_id}: {count} cycles")
    
    return cycles_df, tSC_results


In [9]:
# Run multi-rat tSC pipeline
print("\nRunning multi-rat tSC pipeline...")
cycles_df, tSC_results = run_multi_rat_tSC_pipeline(
        all_rats,
        n_pca=5,
        freq_vec=np.arange(15, 141, 1),
        zscore_features=True,
        mad_k=2.0,
        weighted_alpha=0.4,
        ignore_edge_bins=1
    )
    
# Save results
output_dir = Path("./multi_rat_tSC_results")
output_dir.mkdir(parents=True, exist_ok=True)
    
# Save cycles dataframe
cycles_df.to_csv(output_dir / "cycles_df_all_rats.csv", index=False)
print(f"\nSaved cycles_df to {output_dir / 'cycles_df_all_rats.csv'}")
    
# Save tSC results as pickle
with open(output_dir / "tSC_results_all_rats.pkl", "wb") as f:
    pickle.dump(tSC_results, f, protocol=pickle.HIGHEST_PROTOCOL)
print(f"Saved tSC_results to {output_dir / 'tSC_results_all_rats.pkl'}")
    
print("\n✅ Analysis complete!")


Running multi-rat tSC pipeline...
Combining data from all rats...
  Total phasic segments: 1378
  Total tonic segments: 2210


Total cycles combined: 417842
  Phasic: 21721
  Tonic: 396121
Running PCA...
Running ICA...

=== Multi-Rat PCA/ICA Complete ===
Total cycles: 417842
Mode — featZ (median): 45.00 Hz
Mode — featZ strong (median): 54.00 Hz
Mode — featZ 5Hz reflect strong (median): 66.00 Hz
Mode — featZ 10Hz reflect strong (median): 72.00 Hz

Per-rat cycle counts:
  Rat 1: 51844 cycles
  Rat 3: 45356 cycles
  Rat 4: 53116 cycles
  Rat 6: 101393 cycles
  Rat 9: 65162 cycles
  Rat 11: 61171 cycles
  Rat 13: 39800 cycles

Saved cycles_df to multi_rat_tSC_results/cycles_df_all_rats.csv
Saved tSC_results to multi_rat_tSC_results/tSC_results_all_rats.pkl

✅ Analysis complete!


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

# try UMAP; fallback to t-SNE if not available
try:
    import umap
    _HAS_UMAP = True
except Exception:
    _HAS_UMAP = False
    from sklearn.manifold import TSNE

def _coerce_keys(df: pd.DataFrame) -> pd.DataFrame:
    """Make sure rat_id is string and cycle_type is lowercased string."""
    out = df.copy()
    # rat_id → string (preserves IDs like '1', '01', etc. safely)
    if "rat_id" in out.columns:
        out["rat_id"] = out["rat_id"].astype(str)
    # cycle_type → lower string ('phasic'/'tonic')
    if "cycle_type" in out.columns:
        out["cycle_type"] = out["cycle_type"].astype(str).str.lower()
    return out

def _group_order_index(df: pd.DataFrame, group_keys):
    """Return 0..n-1 index per group in current row order."""
    g = df[group_keys].copy().fillna("__NA__")
    return g.groupby(group_keys).cumcount()

def _safe_sort(df: pd.DataFrame, prefer_order=("rat_id","cycle_type","date","SD","trial","interval_idx","cycle_idx_in_interval")):
    keys = [k for k in prefer_order if k in df.columns]
    return df.sort_values(keys, kind="mergesort") if keys else df

def plot_umap_all_rats_waveforms_from_loaded(
    all_rats_data: dict,
    cycles_df: pd.DataFrame,
    mode_col: str = "mode_freq_hz_featZ_smooth",  # or 'mode_freq_hz_featZ_smooth5_pad'
    scale_features: bool = True,
    n_neighbors: int = 15,
    min_dist: float = 0.1,
    random_state: int = 42,
    small_multiples: bool = True,
):
    # -------- 1) Concatenate waveforms --------
    wf_frames = []
    for rid, payload in all_rats_data.items():
        wf = payload.get("waveforms_df")
        if wf is None or len(wf) == 0:
            continue
        w = wf.copy()
        if "rat_id" not in w.columns:
            w["rat_id"] = str(rid)  # ensure string
        wf_frames.append(w)

    if not wf_frames:
        raise RuntimeError("No waveforms_df found in all_rats_data.")

    waveforms_all = pd.concat(wf_frames, ignore_index=True)

    # -------- 2) Coerce key dtypes consistently --------
    waveforms_all = _coerce_keys(waveforms_all)
    cycles_all    = _coerce_keys(cycles_df)

    if "rat_id" not in cycles_all.columns or "cycle_type" not in cycles_all.columns:
        raise RuntimeError("cycles_df must contain 'rat_id' and 'cycle_type'.")

    if mode_col not in cycles_all.columns:
        raise RuntimeError(f"`{mode_col}` not found in cycles_df. Available: {list(cycles_all.columns)}")

    # -------- 3) Stable sort then build gidx in (rat_id, cycle_type) ----------
    waveforms_all = _safe_sort(waveforms_all)
    cycles_all    = _safe_sort(cycles_all)

    waveforms_all["gidx"] = _group_order_index(waveforms_all, ["rat_id", "cycle_type"])
    cycles_all["gidx"]    = _group_order_index(cycles_all,    ["rat_id", "cycle_type"])

    # -------- 4) Merge color onto waveforms by (rat_id, cycle_type, gidx) -----
    df_color = cycles_all[["rat_id","cycle_type","gidx", mode_col]].copy()
    # Use 'how="inner"' without validate, because cycles can legitimately be filtered upstream
    df_plot  = waveforms_all.merge(df_color, on=["rat_id","cycle_type","gidx"], how="inner")

    if df_plot.empty:
        raise RuntimeError("No overlap between waveforms_all and cycles_df after alignment. "
                           "Check that rat_id/cycle_type exist in both and orders match.")

    # -------- 5) Build feature matrix -----------------------------------------
    numeric_cols = df_plot.select_dtypes(include=[np.number]).columns.tolist()
    # remove metadata and non-waveform numeric columns from features
    meta_cols = {
        "rat_id","condition","trial","cycle_type","SD","date",
        "interval_idx","cycle_idx_in_interval","gidx"
    }
    # also remove any tSC- or mode-related numeric columns if present
    meta_cols |= {c for c in df_plot.columns if c.startswith("tSC") or c.startswith("mode_freq_hz_")}
    feature_cols = [c for c in numeric_cols if c not in meta_cols]
    if not feature_cols:
        raise RuntimeError("No numeric waveform feature columns found for embedding.")

    X = df_plot[feature_cols].to_numpy()
    if scale_features:
        X = StandardScaler().fit_transform(X)

    color = df_plot[mode_col].to_numpy().astype(float)
    valid = np.isfinite(color) & np.isfinite(X).all(axis=1)
    df_plot = df_plot.loc[valid].reset_index(drop=True)
    X = X[valid]
    color = color[valid]

    # -------- 6) Compute embedding --------------------------------------------
    if _HAS_UMAP:
        reducer = umap.UMAP(n_neighbors=n_neighbors, min_dist=min_dist,
                            n_components=2, random_state=random_state)
        emb = reducer.fit_transform(X)
        xlab, ylab, method = "UMAP-1", "UMAP-2", "UMAP"
    else:
        tsne = TSNE(n_components=2, perplexity=30, learning_rate='auto',
                    init='pca', random_state=random_state)
        emb = tsne.fit_transform(X)
        xlab, ylab, method = "Dim 1", "Dim 2", "t-SNE"

    # -------- 7) Plot ---------------------------------------------------------
    phasic_m = (df_plot["cycle_type"].values == "phasic")
    tonic_m  = (df_plot["cycle_type"].values == "tonic")

    vmin = np.nanpercentile(color, 1)
    vmax = np.nanpercentile(color, 99)

    plt.figure(figsize=(10, 6))
    sc1 = plt.scatter(emb[tonic_m, 0], emb[tonic_m, 1],
                      c=color[tonic_m], cmap="hot", vmin=vmin, vmax=vmax,
                      s=28, alpha=0.80, edgecolors='none', label="Tonic")
    plt.scatter(emb[phasic_m, 0], emb[phasic_m, 1],
                c=color[phasic_m], cmap="hot", vmin=vmin, vmax=vmax,
                s=42, marker='X', edgecolors='k', linewidths=0.25, label="Phasic")
    plt.xlabel(xlab); plt.ylabel(ylab)
    plt.title(f"All-rats {method} of waveform features — color: {mode_col} (Hz)")
    cbar = plt.colorbar(sc1)
    cbar.set_label(f"{mode_col} (Hz)")
    plt.legend(frameon=False)
    plt.tight_layout()
    plt.show()

    # -------- 8) Optional per-rat small multiples -----------------------------
    if small_multiples:
        rats = sorted(df_plot["rat_id"].unique())
        ncols, nrows = 3, int(np.ceil(len(rats)/3))
        fig, axes = plt.subplots(nrows, ncols, figsize=(4.2*ncols, 3.8*nrows), squeeze=False)
        for i, rid in enumerate(rats):
            ax = axes[i//ncols, i % ncols]
            m = (df_plot["rat_id"].values == rid)
            sc = ax.scatter(emb[m, 0], emb[m, 1],
                            c=color[m], cmap="hot", vmin=vmin, vmax=vmax,
                            s=30, alpha=0.85, edgecolors='none')
            ax.set_title(f"Rat {rid}")
            ax.set_xlabel(xlab); ax.set_ylabel(ylab)
        for j in range(len(rats), nrows*ncols):
            axes[j//ncols, j % ncols].axis('off')
        fig.colorbar(sc, ax=axes, shrink=0.95, label=f"{mode_col} (Hz)")
        fig.suptitle(f"Per-rat {method} (waveform features), color: {mode_col}", y=1.02)
        plt.tight_layout()
        plt.show()


plot_umap_all_rats_waveforms_from_loaded(
    all_rats_data=all_rats,
    cycles_df=cycles_df,
    mode_col="mode_freq_hz_featZ_smooth",  # or "mode_freq_hz_featZ_smooth5_pad"
    scale_features=True,
    n_neighbors=15,
    min_dist=0.1,
    random_state=42,
    small_multiples=True,
)

OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

# try UMAP; fallback to t-SNE if not available
try:
    import umap
    _HAS_UMAP = True
except Exception:
    _HAS_UMAP = False
    from sklearn.manifold import TSNE

def _coerce_keys(df: pd.DataFrame) -> pd.DataFrame:
    """Make sure rat_id is string and cycle_type is lowercased string."""
    out = df.copy()
    # rat_id → string (preserves IDs like '1', '01', etc. safely)
    if "rat_id" in out.columns:
        out["rat_id"] = out["rat_id"].astype(str)
    # cycle_type → lower string ('phasic'/'tonic')
    if "cycle_type" in out.columns:
        out["cycle_type"] = out["cycle_type"].astype(str).str.lower()
    return out

def _group_order_index(df: pd.DataFrame, group_keys):
    """Return 0..n-1 index per group in current row order."""
    g = df[group_keys].copy().fillna("__NA__")
    return g.groupby(group_keys).cumcount()

def _safe_sort(df: pd.DataFrame, prefer_order=("rat_id","cycle_type","date","SD","trial","interval_idx","cycle_idx_in_interval")):
    keys = [k for k in prefer_order if k in df.columns]
    return df.sort_values(keys, kind="mergesort") if keys else df

def plot_umap_all_rats_waveforms_from_loaded(
    all_rats_data: dict,
    cycles_df: pd.DataFrame,
    mode_col: str = "mode_freq_hz_featZ_smooth",  # or 'mode_freq_hz_featZ_smooth5_pad'
    scale_features: bool = True,
    n_neighbors: int = 15,
    min_dist: float = 0.1,
    random_state: int = 42,
    small_multiples: bool = True,
):
    # -------- 1) Concatenate waveforms --------
    wf_frames = []
    for rid, payload in all_rats_data.items():
        wf = payload.get("waveforms_df")
        if wf is None or len(wf) == 0:
            continue
        w = wf.copy()
        if "rat_id" not in w.columns:
            w["rat_id"] = str(rid)  # ensure string
        wf_frames.append(w)

    if not wf_frames:
        raise RuntimeError("No waveforms_df found in all_rats_data.")

    waveforms_all = pd.concat(wf_frames, ignore_index=True)

    # -------- 2) Coerce key dtypes consistently --------
    waveforms_all = _coerce_keys(waveforms_all)
    cycles_all    = _coerce_keys(cycles_df)

    if "rat_id" not in cycles_all.columns or "cycle_type" not in cycles_all.columns:
        raise RuntimeError("cycles_df must contain 'rat_id' and 'cycle_type'.")

    if mode_col not in cycles_all.columns:
        raise RuntimeError(f"`{mode_col}` not found in cycles_df. Available: {list(cycles_all.columns)}")

    # -------- 3) Stable sort then build gidx in (rat_id, cycle_type) ----------
    waveforms_all = _safe_sort(waveforms_all)
    cycles_all    = _safe_sort(cycles_all)

    waveforms_all["gidx"] = _group_order_index(waveforms_all, ["rat_id", "cycle_type"])
    cycles_all["gidx"]    = _group_order_index(cycles_all,    ["rat_id", "cycle_type"])

    # -------- 4) Merge color onto waveforms by (rat_id, cycle_type, gidx) -----
    df_color = cycles_all[["rat_id","cycle_type","gidx", mode_col]].copy()
    df_plot  = waveforms_all.merge(df_color, on=["rat_id","cycle_type","gidx"], how="inner")

    if df_plot.empty:
        raise RuntimeError("No overlap between waveforms_all and cycles_df after alignment. "
                           "Check that rat_id/cycle_type exist in both and orders match.")

    # -------- 5) Build feature matrix -----------------------------------------
    numeric_cols = df_plot.select_dtypes(include=[np.number]).columns.tolist()
    meta_cols = {
        "rat_id","condition","trial","cycle_type","SD","date",
        "interval_idx","cycle_idx_in_interval","gidx"
    }
    meta_cols |= {c for c in df_plot.columns if c.startswith("tSC") or c.startswith("mode_freq_hz_")}
    feature_cols = [c for c in numeric_cols if c not in meta_cols]
    if not feature_cols:
        raise RuntimeError("No numeric waveform feature columns found for embedding.")

    X = df_plot[feature_cols].to_numpy()
    if scale_features:
        X = StandardScaler().fit_transform(X)

    color = df_plot[mode_col].to_numpy().astype(float)
    valid = np.isfinite(color) & np.isfinite(X).all(axis=1)
    df_plot = df_plot.loc[valid].reset_index(drop=True)
    X = X[valid]
    color = color[valid]

    # -------- 6) Compute embedding --------------------------------------------
    if _HAS_UMAP:
        reducer = umap.UMAP(n_neighbors=n_neighbors, min_dist=min_dist,
                            n_components=2, random_state=random_state)
        emb = reducer.fit_transform(X)
        xlab, ylab, method = "UMAP-1", "UMAP-2", "UMAP"
    else:
        tsne = TSNE(n_components=2, perplexity=30, learning_rate='auto',
                    init='pca', random_state=random_state)
        emb = tsne.fit_transform(X)
        xlab, ylab, method = "Dim 1", "Dim 2", "t-SNE"

    # -------- 7) Plot (single combined)  --------------------------------------
    from matplotlib.colors import Normalize
    from matplotlib.ticker import MaxNLocator, StrMethodFormatter
    import matplotlib.cm as cm
    import matplotlib.pyplot as plt

    phasic_m = (df_plot["cycle_type"].values == "phasic")
    tonic_m  = (df_plot["cycle_type"].values == "tonic")

    # robust color limits + shared norm
    vmin = float(np.nanpercentile(color, 1))
    vmax = float(np.nanpercentile(color, 99))
    norm = Normalize(vmin=vmin, vmax=vmax)
    cmap = "hot"

    fig, ax = plt.subplots(figsize=(10, 6), dpi=160)

    sc1 = ax.scatter(
        emb[tonic_m, 0], emb[tonic_m, 1],
        c=color[tonic_m], cmap=cmap, norm=norm,
        s=26, alpha=0.80, linewidths=0, label="Tonic"
    )
    ax.scatter(
        emb[phasic_m, 0], emb[phasic_m, 1],
        c=color[phasic_m], cmap=cmap, norm=norm,
        s=40, marker='X', edgecolors='k', linewidths=0.25, alpha=0.95, label="Phasic"
    )

    ax.set_xlabel(xlab); ax.set_ylabel(ylab)
    ax.set_title(f"All-rats {method} of waveform features — color: {mode_col} (Hz)")
    ax.legend(frameon=False)

    # slim, tall colorbar on the right (clean ticks)
    fig.subplots_adjust(right=0.92)
    cax = fig.add_axes([0.93, 0.16, 0.02, 0.70])  # [left, bottom, width, height]
    sm = cm.ScalarMappable(norm=norm, cmap=cmap); sm.set_array([])
    cb = fig.colorbar(sm, cax=cax)
    cb.ax.yaxis.set_major_locator(MaxNLocator(6))
    cb.ax.yaxis.set_major_formatter(StrMethodFormatter("{x:.0f}"))
    cb.outline.set_linewidth(0.6)
    cb.set_label(f"{mode_col} (Hz)", rotation=90, labelpad=8)

    plt.tight_layout(rect=[0.02, 0.02, 0.91, 0.97])
    plt.show()

    # -------- 8) Optional per-rat small multiples -----------------------------
    if small_multiples:
        rats = sorted(df_plot["rat_id"].unique())
        ncols, nrows = 3, int(np.ceil(len(rats)/3))
        fig, axes = plt.subplots(nrows, ncols, figsize=(4.4*ncols, 3.9*nrows), squeeze=False, dpi=160)

        last_sc = None
        for i, rid in enumerate(rats):
            ax = axes[i//ncols, i % ncols]
            m = (df_plot["rat_id"].values == rid)
            last_sc = ax.scatter(
                emb[m, 0], emb[m, 1],
                c=color[m], cmap=cmap, norm=norm,
                s=28, alpha=0.85, linewidths=0
            )
            ax.set_title(f"Rat {rid}", fontsize=10)
            ax.set_xlabel(xlab); ax.set_ylabel(ylab)

        # blank unused axes
        for j in range(len(rats), nrows*ncols):
            axes[j//ncols, j % ncols].axis('off')

        # shared, tall colorbar on the right of grid
        fig.subplots_adjust(right=0.92)
        cax = fig.add_axes([0.93, 0.15, 0.02, 0.72])
        cb = fig.colorbar(last_sc, cax=cax)
        cb.ax.yaxis.set_major_locator(MaxNLocator(6))
        cb.ax.yaxis.set_major_formatter(StrMethodFormatter("{x:.0f}"))
        cb.outline.set_linewidth(0.6)
        cb.set_label(f"{mode_col} (Hz)")

        fig.suptitle(f"Per-rat {method} (waveform features), color: {mode_col}", y=0.99)
        plt.tight_layout(rect=[0.03, 0.03, 0.91, 0.97])
        plt.show()

    # -------- 8) Optional per-rat small multiples -----------------------------
    if small_multiples:
        rats = sorted(df_plot["rat_id"].unique())
        ncols, nrows = 3, int(np.ceil(len(rats)/3))
        fig, axes = plt.subplots(nrows, ncols, figsize=(4.2*ncols, 3.8*nrows), squeeze=False)
        for i, rid in enumerate(rats):
            ax = axes[i//ncols, i % ncols]
            m = (df_plot["rat_id"].values == rid)
            sc = ax.scatter(emb[m, 0], emb[m, 1],
                            c=color[m], cmap="hot", vmin=vmin, vmax=vmax,
                            s=30, alpha=0.85, edgecolors='none')
            ax.set_title(f"Rat {rid}")
            ax.set_xlabel(xlab); ax.set_ylabel(ylab)
        for j in range(len(rats), nrows*ncols):
            axes[j//ncols, j % ncols].axis('off')
        fig.colorbar(sc, ax=axes, shrink=0.95, label=f"{mode_col} (Hz)")
        fig.suptitle(f"Per-rat {method} (waveform features), color: {mode_col}", y=1.02)
        plt.tight_layout()
        plt.show()


plot_umap_all_rats_waveforms_from_loaded(
    all_rats_data=all_rats,
    cycles_df=cycles_df,
    mode_col="mode_freq_hz_featZ_smooth",  # or "mode_freq_hz_featZ_smooth5_pad"
    scale_features=True,
    n_neighbors=15,
    min_dist=0.1,
    random_state=42,
    small_multiples=True,
)